# 08 — Duplicate Feature Group Analysis

## Objective

Analyze duplicate feature groups and target-price variation without introducing data leakage.

This notebook investigates whether identical feature combinations:

- occur multiple times,
- have identical or different target values,
- could contaminate a conventional random train/test split.

### Feature group

A feature group is defined by:

- `bhk`
- `propertytype`
- `location`
- `sqft`

### Target

- `totalprice`

This notebook is diagnostic only.

No model training or preprocessing is performed here.

In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [20]:
df = pd.read_csv("../data/processed/house_prices_refined.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (13297, 5)

Columns:
['bhk', 'propertytype', 'location', 'sqft', 'totalprice']


In [21]:
feature_columns = [
    "bhk",
    "propertytype",
    "location",
    "sqft"
]

target_column = "totalprice"

print("Feature columns:", feature_columns)
print("Target column:", target_column)

Feature columns: ['bhk', 'propertytype', 'location', 'sqft']
Target column: totalprice


In [22]:
required_columns = feature_columns + [target_column]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("All required columns are present.")

All required columns are present.


## 1. Create Feature Groups

A feature group represents rows having the same values for:

`bhk + propertytype + location + sqft`

The target `totalprice` is intentionally excluded from the group definition.

This allows us to check whether identical input features can have different target values.

In [23]:
df["feature_group"] = (
    df[feature_columns]
    .astype(str)
    .agg("||".join, axis=1)
)

print("Total rows:", len(df))
print("Unique feature groups:", df["feature_group"].nunique())

Total rows: 13297
Unique feature groups: 9359


In [24]:
group_counts = df["feature_group"].value_counts()

duplicate_groups = group_counts[group_counts > 1]

print("Total feature groups:", len(group_counts))
print("Duplicate feature groups:", len(duplicate_groups))
print("Rows belonging to duplicate groups:", duplicate_groups.sum())

Total feature groups: 9359
Duplicate feature groups: 1817
Rows belonging to duplicate groups: 5755


In [25]:
print("Duplicate group size distribution:")
print(duplicate_groups.value_counts().sort_index())

Duplicate group size distribution:
count
2     1044
3      338
4      152
5       96
6       75
7       33
8       24
9       12
10       9
11       6
12       7
13       5
14       4
15       3
16       2
17       2
19       1
23       3
24       1
Name: count, dtype: int64


## 2. Analyze Target Variation

For each duplicate feature group, we compare the corresponding `totalprice` values.

There are two important cases:

1. Same features + same target
2. Same features + different target

Different target values do not automatically mean leakage.

They indicate that the available four features do not completely explain the target price.

In [26]:
group_target_stats = (
    df.groupby("feature_group")[target_column]
    .agg(
        count="count",
        unique_targets="nunique",
        min_target="min",
        max_target="max",
        mean_target="mean",
        median_target="median",
        std_target="std"
    )
)

duplicate_group_stats = group_target_stats[
    group_target_stats["count"] > 1
].copy()

print("Duplicate feature groups:", len(duplicate_group_stats))

Duplicate feature groups: 1817


In [27]:
same_target_groups = duplicate_group_stats[
    duplicate_group_stats["unique_targets"] == 1
]

varying_target_groups = duplicate_group_stats[
    duplicate_group_stats["unique_targets"] > 1
]

print("Duplicate groups with identical targets:",
      len(same_target_groups))

print("Duplicate groups with varying targets:",
      len(varying_target_groups))

Duplicate groups with identical targets: 0
Duplicate groups with varying targets: 1817


In [28]:
total_duplicate_groups = len(duplicate_group_stats)

same_target_percentage = (
    len(same_target_groups) / total_duplicate_groups * 100
)

varying_target_percentage = (
    len(varying_target_groups) / total_duplicate_groups * 100
)

print(f"Identical-target groups: {same_target_percentage:.2f}%")
print(f"Varying-target groups: {varying_target_percentage:.2f}%")

Identical-target groups: 0.00%
Varying-target groups: 100.00%


In [29]:
duplicate_group_stats["cv"] = np.where(
    duplicate_group_stats["mean_target"] != 0,
    duplicate_group_stats["std_target"] /
    duplicate_group_stats["mean_target"],
    np.nan
)

print("Median within-group coefficient of variation:",
      duplicate_group_stats["cv"].median())

Median within-group coefficient of variation: 0.2736984114894404


In [30]:
largest_variation = (
    duplicate_group_stats
    .sort_values("cv", ascending=False)
    .head(10)
)

largest_variation[
    [
        "count",
        "unique_targets",
        "min_target",
        "max_target",
        "mean_target",
        "median_target",
        "cv"
    ]
]

,count,unique_targets,min_target,max_target,mean_target,median_target,cv
feature_group,,,,,,,
3||House||Ludhiana||1800,6,6,5200000,80000000,1.878333e+07,6750000.0,1.598031
2||Flat||NewDelhi||1100,7,7,11500000,175000000,3.824286e+07,16500000.0,1.578652
3||House||Ernakulam||1650,2,2,590000,11900000,6.245000e+06,6245000.0,1.280605
3||House||Amritsar||2500,3,3,5000000,60000000,2.433333e+07,8000000.0,1.270876
3||Flat||Faridabad||2000,2,2,1500000,16000000,8.750000e+06,8750000.0,1.171777
2||Flat||Kochi||1817,2,2,4050000,41000000,2.252500e+07,22525000.0,1.159938
3||Villa||Jodhpur||3000,2,2,900000,9000000,4.950000e+06,4950000.0,1.157084
3||Flat||Dhanbad||1100,2,2,400000,4000000,2.200000e+06,2200000.0,1.157084
2||House||Madurai||1495,5,5,4800000,45000000,1.482200e+07,8300000.0,1.143273


In [31]:
varying_group_keys = varying_target_groups.index[:10]

examples = (
    df[df["feature_group"].isin(varying_group_keys)]
    .sort_values("feature_group")
)

examples[
    feature_columns + [target_column, "feature_group"]
]

,bhk,propertytype,location,sqft,totalprice,feature_group
8333,1,Flat,Lucknow,610,2900000,1||Flat||Lucknow||610
8402,1,Flat,Lucknow,610,2800000,1||Flat||Lucknow||610
8808,1,Flat,Madurai,1200,680000,1||Flat||Madurai||1200
8951,1,Flat,Madurai,1200,180000,1||Flat||Madurai||1200
9177,1,Flat,Mangalore,645,2800000,1||Flat||Mangalore||645
9230,1,Flat,Mangalore,645,2180000,1||Flat||Mangalore||645
9674,1,Flat,Mysore,1000,7000000,1||Flat||Mysore||1000
9597,1,Flat,Mysore,1000,6700000,1||Flat||Mysore||1000
9598,1,Flat,Mysore,1070,5500000,1||Flat||Mysore||1070
9434,1,Flat,Mysore,1070,5200000,1||Flat||Mysore||1070


## 3. Demonstrate Random-Split Group Contamination

A conventional random train/test split can place rows from the same feature group into both datasets.

This creates overlap between train and test feature groups.

The model may therefore encounter the same feature combination during training that appears again in the test set.

This notebook measures that contamination only.

No model is trained.

In [32]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (10637, 6)
Test shape: (2660, 6)


In [33]:
train_groups = set(train_df["feature_group"])
test_groups = set(test_df["feature_group"])

overlapping_groups = train_groups.intersection(test_groups)

print("Unique feature groups in train:", len(train_groups))
print("Unique feature groups in test:", len(test_groups))
print("Overlapping feature groups:", len(overlapping_groups))

Unique feature groups in train: 7789
Unique feature groups in test: 2363
Overlapping feature groups: 793


In [34]:
test_contaminated_mask = test_df["feature_group"].isin(
    overlapping_groups
)

contaminated_test_rows = test_contaminated_mask.sum()

contamination_percentage = (
    contaminated_test_rows / len(test_df) * 100
)

print("Contaminated test rows:", contaminated_test_rows)
print(f"Test contamination: {contamination_percentage:.2f}%")

Contaminated test rows: 1039
Test contamination: 39.06%


## 4. Interpretation

The duplicate feature groups are not automatically target leakage.

The analysis shows that identical feature combinations can have different `totalprice` values. This means the four available features do not fully explain every observed price.

The main evaluation problem is split contamination:

- An ordinary random split can place the same feature group in both train and test.
- This makes the test set less independent from the training set.
- A group-aware split keeps each feature group entirely within either train or test.

Therefore, model evaluation uses a `GroupShuffleSplit` based on:

`bhk + propertytype + location + sqft`

Notebook 09 verifies that the grouped split produces zero feature-group overlap.

In [35]:
summary = pd.DataFrame({
    "Metric": [
        "Total rows",
        "Unique feature groups",
        "Duplicate feature groups",
        "Rows in duplicate groups",
        "Duplicate groups with identical targets",
        "Duplicate groups with varying targets",
        "Median within-group CV",
        "Random-split overlapping groups",
        "Random-split contaminated test rows",
        "Random-split test contamination %"
    ],
    "Value": [
        len(df),
        df["feature_group"].nunique(),
        len(duplicate_groups),
        duplicate_groups.sum(),
        len(same_target_groups),
        len(varying_target_groups),
        duplicate_group_stats["cv"].median(),
        len(overlapping_groups),
        contaminated_test_rows,
        contamination_percentage
    ]
})

summary

,Metric,Value
0,Total rows,13297.000000
1,Unique feature groups,9359.000000
2,Duplicate feature groups,1817.000000
3,Rows in duplicate groups,5755.000000
4,Duplicate groups with identical targets,0.000000
5,Duplicate groups with varying targets,1817.000000
6,Median within-group CV,0.273698
7,Random-split overlapping groups,793.000000
8,Random-split contaminated test rows,1039.000000
9,Random-split test contamination %,39.060150


# Conclusion

The duplicate-group analysis confirms that repeated feature combinations exist in the refined dataset.

Many duplicate feature groups contain different `totalprice` values, showing that the current feature set does not uniquely determine the target.

The critical issue is not that duplicate observations are themselves target leakage.

The critical issue is that a conventional random train/test split can distribute the same feature group across both datasets.

Therefore, group-aware splitting is required for a more independent evaluation.

For this project, the feature group is:

`bhk + propertytype + location + sqft`

Notebook 09 implements and verifies the deterministic `GroupShuffleSplit` used for model evaluation.